In [1]:
import glob
import os
import numpy as np
import pickle
import sys
import torch


print(f"numpy 版本: {np.__version__}")

motions_dir_pth = '/home/winky/Documents/code/humanoid/TWIST2_full/OMOMO_g1_GMR'

# 获取目录下所有 .pkl 和 .npz 文件路径
motion_files_pkl = glob.glob(os.path.join(motions_dir_pth, '*.pkl'))

motion_files = sorted(motion_files_pkl )


numpy 版本: 1.24.0


In [2]:
import numpy as np, importlib.util
print("numpy:", np.__version__)
print("numpy file:", np.__file__)
print("has numpy._core ?", importlib.util.find_spec("numpy._core") is not None)


numpy: 1.24.0
numpy file: /home/winky/anaconda3/envs/robot_lab_winky/lib/python3.11/site-packages/numpy/__init__.py
has numpy._core ? False


In [3]:
import pickle

class NumpyCompatUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        # numpy 2.x -> 1.x 兼容
        if module.startswith("numpy._core"):
            module = "numpy.core" + module[len("numpy._core"):]
        return super().find_class(module, name)

def load_pickle_numpy_compat(path: str):
    with open(path, "rb") as f:
        return NumpyCompatUnpickler(f).load()

In [4]:
print(f"找到 {len(motion_files)} 个 motion 文件:")

找到 5882 个 motion 文件:


In [5]:
curr_file = motion_files[0]
# with open(curr_file, "rb") as f:
motion_data = load_pickle_numpy_compat(curr_file)

fps = motion_data["fps"]
print(f"fps: {fps}")


fps: 30


In [6]:
print("the key of motion_data:")
print(motion_data.keys())

print("root_pos shape:", motion_data['root_pos'].shape)
print("dof_pos shape:", motion_data['dof_pos'].shape)
print("dof_pos 0:", motion_data['dof_pos'][0])

print("local_body_pos shape:", motion_data['local_body_pos'].shape)
print("local_body_pos 0:", motion_data['local_body_pos'][0])


print("link_body_list:", motion_data['link_body_list'])


the key of motion_data:
dict_keys(['fps', 'root_pos', 'root_rot', 'dof_pos', 'local_body_pos', 'link_body_list'])
root_pos shape: (132, 3)
dof_pos shape: (132, 29)
dof_pos 0: [-0.25279107 -0.10897656 -0.01158752  0.2437885  -0.28947382  0.15707281
 -0.33381044 -0.06612755 -0.09034693  0.56662652 -0.40794344  0.016508
 -0.08710816 -0.06234248 -0.0570128  -0.1611132   0.32511235 -0.44798821
  0.99493751  0.30404191 -0.02033761 -0.04661259 -0.00247624 -0.49261342
 -0.05117355 -0.03156436 -0.12305071 -0.1385715  -0.26565032]
local_body_pos shape: (132, 38, 3)
local_body_pos 0: [[ 0.0000000e+00  0.0000000e+00  0.0000000e+00]
 [ 0.0000000e+00  6.4452000e-02 -1.0270000e-01]
 [ 7.6195188e-03  1.1645200e-01 -1.3219677e-01]
 [ 8.1544623e-02  1.0295257e-01 -2.3409727e-01]
 [ 8.3606347e-02  8.6702473e-02 -4.2726177e-01]
 [ 8.5698761e-02  5.8202535e-02 -7.2590840e-01]
 [ 9.0820432e-02  5.6334887e-02 -7.4259865e-01]
 [ 1.9669816e-01  5.5556200e-02 -7.4638784e-01]
 [ 0.0000000e+00  0.0000000e+00  0.0

In [7]:
root_pos = motion_data['root_pos']
num_frames = root_pos.shape[0]

print("num_frames:", num_frames)
motion_len_s = 1.0 / fps * (num_frames - 1)
print("motion_len_s:", motion_len_s)



num_frames: 132
motion_len_s: 4.366666666666666


In [8]:
_device = 'cuda:0'

root_pos = torch.tensor(motion_data["root_pos"], 
                        dtype=torch.float, 
                        device=_device)
local_body_pos = torch.tensor(motion_data["local_body_pos"], 
                              dtype=torch.float, 
                              device=_device)

# 脚对齐到地面
body_pos = local_body_pos + root_pos.unsqueeze(1)
lowest_body_part = torch.min(body_pos[..., 2])

print("lowest_body_part:", lowest_body_part)
# adjust the height of the root position
root_pos[..., 2] -= lowest_body_part

lowest_body_part: tensor(0.0174, device='cuda:0')


ModuleNotFoundError: No module named 'pxr'
```bash
pip install usd-core
```

In [20]:
from robot_lab.utils.isaacgym_torch_utils import quat_rotate_inverse
from robot_lab.utils.torch_utils import euler_from_quaternion, quat_diff

from robot_lab.utils.motion_lib.motion_math_utils import compute_so3_derivative

In [22]:
# 加载 root_rot（四元数旋转）并转换为 torch tensor
root_rot = torch.tensor(motion_data['root_rot'], 
                        dtype=torch.float, 
                        device=_device)

dof_pos = torch.tensor(motion_data['dof_pos'], 
                      dtype=torch.float, 
                      device=_device)

dt = 1.0 / fps 
print("时间步长 dt:", dt)

total_time = (num_frames - 1) * dt
print("总时间 total_time:", total_time)


root_pos_delta = root_pos[-1] - root_pos[0]
root_pos_delta[..., -1] = 0
print("root_pos_delta 整个动作的根位置总位移, 忽略高度:", root_pos_delta)


root_vel = torch.gradient(root_pos, spacing=dt, dim=0)[0]
print("root_vel 根位置速度, 打印前2帧:", root_vel[:2])

# 每帧的根位置增量（局部坐标系）
root_pos_delta_local = torch.zeros_like(root_pos)
# cur frame delta pos = cur frame pos - last frame pos
root_pos_delta_local[1:, :] = root_pos[1:, :] - root_pos[:-1, :] 
# first frame delta pos = 0
root_pos_delta_local[0, :] = 0.0 
# 相对上一帧的根位置增量（局部坐标系）
root_pos_delta_local[1:, :] = quat_rotate_inverse(root_rot[:-1, :], 
                                                  root_pos_delta_local[1:, :]) # rotate the delta pos to local frame via last frame rot

print("root_pos_delta_local 相对上一帧的根位置增量（局部坐标系）:")
print(root_pos_delta_local[:2])


# compute the delta rot per frame
root_rot_delta_local = torch.zeros_like(root_pos)
root_rot_delta_local[1:, :] = \
                     euler_from_quaternion(quat_diff(root_rot[1:, :], 
                     root_rot[:-1, :])) # cur frame delta rot = cur frame rot - last frame rot
root_rot_delta_local[0, :] = 0.0
root_rot_delta_local[1:, :] = \
                      quat_rotate_inverse(root_rot[:-1, :], 
                      root_rot_delta_local[1:, :]) # rotate the delta rot to local frame via last frame rot

print("root_rot_delta_local 相对上一帧的根位置增量（局部坐标系）:")
print(root_rot_delta_local[:2])

root_ang_vel = compute_so3_derivative(root_rot, dt)
print("root_ang_vel 根角速度, 打印前2帧:", root_ang_vel[:2])

dof_vel = torch.gradient(dof_pos, spacing=dt, dim=0)[0]
print("dof_vel 关节速度, 打印前2帧:", dof_vel[:2])


时间步长 dt: 0.03333333333333333
总时间 total_time: 4.366666666666666
root_pos_delta 整个动作的根位置总位移, 忽略高度: tensor([0.0597, 0.0582, 0.0000], device='cuda:0')
root_vel 根位置速度, 打印前2帧: tensor([[-0.0307, -0.1032, -0.0340],
        [-0.0424, -0.0929, -0.0247]], device='cuda:0')
root_pos_delta_local 相对上一帧的根位置增量（局部坐标系）:
tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [-1.5179e-05,  3.5979e-03, -1.1039e-03]], device='cuda:0')
root_rot_delta_local 相对上一帧的根位置增量（局部坐标系）:
tensor([[ 0.0000,  0.0000,  0.0000],
        [-0.0073, -0.0252, -0.0215]], device='cuda:0')
root_ang_vel 根角速度, 打印前2帧: tensor([[-0.5578, -0.6095,  0.6001],
        [-0.3112, -0.3099,  0.3910]], device='cuda:0')
dof_vel 关节速度, 打印前2帧: tensor([[-1.1185, -0.7873, -0.5577,  0.3320,  0.0402,  0.2393, -1.8216, -1.1199,
         -0.5910,  2.1666,  0.1830,  1.0647, -0.6984, -0.3989, -0.0330, -1.7725,
         -0.0439,  0.1183,  1.8334, -0.5744, -1.5516, -0.2298, -1.7126,  0.0327,
          0.1670,  1.9846,  1.1401, -0.2586, -0.9441],
        [-0